In [ ]:
!pip -q install -U transformers accelerate sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

if torch.cuda.is_available():
    device = "cuda"
    print("GPU:", torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("Running on CPU")

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")

if device == "cuda":
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32
    )

model.eval()

print("Model loaded successfully!")
print()
print("LLM Chatbot Ready")
print("Type 'exit' to stop.")
print()

conversation = [
    {
        "role": "system",
        "content": "You are a helpful AI assistant. Answer questions clearly and accurately. If you don't know something, say that you don't know."
    }
]

while True:
    user_message = input("You: ")

    if user_message.lower().strip() in ["exit", "quit", "bye"]:
        print("Goodbye!")
        break

    if not user_message.strip():
        continue

    conversation.append({
        "role": "user",
        "content": user_message
    })

    prompt = tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[1]
    new_tokens = output[0][input_length:]

    assistant_message = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    conversation.append({
        "role": "assistant",
        "content": assistant_message
    })

    print("AI:", assistant_message)
    print()


Running on CPU
Loading tokenizer...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!

LLM Chatbot Ready
Type 'exit' to stop.

You: exit
Goodbye!


In [ ]:
!pip -q install -U transformers accelerate sentencepiece sentence-transformers faiss-cpu pypdf python-docx

import os
import torch
import faiss
import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from docx import Document

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

if torch.cuda.is_available():
    device = "cuda"
    print("GPU:", torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("Running on CPU")

print("Loading LLM...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if device == "cuda":
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32
    )

model.eval()

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL,
    device=device
)

print("Models loaded successfully.")

def load_pdf(path):
    text = ""

    reader = PdfReader(path)

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text

def load_docx(path):
    document = Document(path)

    text = []

    for paragraph in document.paragraphs:
        if paragraph.text.strip():
            text.append(paragraph.text)

    return "\n".join(text)

def load_txt(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def load_file(path):
    extension = os.path.splitext(path)[1].lower()

    if extension == ".pdf":
        return load_pdf(path)

    elif extension == ".docx":
        return load_docx(path)

    elif extension == ".txt":
        return load_txt(path)

    else:
        raise ValueError(
            "Supported files are PDF, DOCX and TXT."
        )

def split_text(text, chunk_size=800, overlap=150):
    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

def create_embeddings(chunks):
    embeddings = embedding_model.encode(
        chunks,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    return embeddings.astype("float32")

def create_faiss_index(embeddings):
    dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(dimension)

    index.add(embeddings)

    return index

def retrieve(query, index, chunks, top_k=3):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_embedding = query_embedding.astype("float32")

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx >= 0 and idx < len(chunks):

            results.append({
                "text": chunks[idx],
                "score": float(score)
            })

    return results

print()
print("Upload your document.")

from google.colab import files

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

print("Uploaded:", file_name)

document_text = load_file(file_name)

print("Document characters:", len(document_text))

chunks = split_text(
    document_text,
    chunk_size=800,
    overlap=150
)

print("Number of chunks:", len(chunks))

embeddings = create_embeddings(chunks)

print("Creating FAISS index...")

index = create_faiss_index(embeddings)

print("RAG database created successfully.")

conversation = [
    {
        "role": "system",
        "content": (
            "You are a helpful AI assistant. "
            "Answer the user's question using the provided context. "
            "If the answer cannot be found in the context, say "
            "'I could not find the answer in the provided document.' "
            "Do not invent information."
        )
    }
]

def generate_answer(question):

    results = retrieve(
        question,
        index,
        chunks,
        top_k=3
    )

    context = "\n\n".join(
        result["text"]
        for result in results
    )

    prompt_messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful AI assistant. "
                "Answer the question using only the provided context. "
                "If the context does not contain the answer, say "
                "'I could not find the answer in the provided document.' "
                "Do not make up information."
            )
        },
        {
            "role": "user",
            "content": (
                "Context:\n\n"
                + context
                + "\n\nQuestion:\n"
                + question
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[1]

    new_tokens = output[0][input_length:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return answer, results

print()
print("RAG chatbot ready.")
print("Ask questions about your document.")
print("Type 'exit' to stop.")
print()

while True:

    question = input("You: ")

    if question.lower().strip() in [
        "exit",
        "quit",
        "bye"
    ]:
        print("Goodbye!")
        break

    if not question.strip():
        continue

    answer, sources = generate_answer(question)

    print()
    print("AI:", answer)

    print()
    print("Retrieved sources:")

    for i, source in enumerate(sources, 1):

        print(
            f"\nSource {i} "
            f"(similarity: {source['score']:.3f})"
        )

        print(source["text"][:500])

    print()
    print("-" * 60)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.5 MB/s eta 0:00:00
Running on CPU
Loading LLM...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

... (10 outputs left out)

Models loaded successfully.

Upload your document.


Saving chatgpt_search_history_summary.pdf to chatgpt_search_history_summary.pdf
Uploaded: chatgpt_search_history_summary.pdf
Document characters: 10568
Number of chunks: 3


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Creating FAISS index...
RAG database created successfully.

RAG chatbot ready.
Ask questions about your document.
Type 'exit' to stop.


AI: Computer Networks

Retrieved sources:

Source 1 (similarity: 0.312)
and examples of Priority Scheduling using processes P1–P5. 19–22 May 2024 Computer Networks – core topics Requested OSI model with a neat diagram and TCP/IP reference model. 22 May 2024 Probability & Queueing / PSQT Identified important PSQT Unit-3 topics: Pollaczek–Khinchine (PK) formula and Model-I sums. 24–25 May 2024 Wireless and Mobile Communication Requested detailed LTE architecture with a suitable diagram, including an 8-mark format. 24 May 2024 Image editing / creative requests Requeste

Source 2 (similarity: 0.274)
probability/queueing problems, including P(X=2). 24–25 May 2024 — Wireless and Mobile Communication  Requested detailed LTE architecture with a suitable diagram, including an 8-mark format.  Requested detailed UMTS architecture with a neat sketch.  Requeste

In [ ]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments
from trl import SFTTrainer

# 1. Convert document chunks to dataset
training_data = [
    {
        "messages": [
            {"role": "system", "content": "You are an expert assistant trained on target document context."},
            {"role": "user", "content": "Summarize or extract key information from this passage."},
            {"role": "assistant", "content": chunk}
        ]
    }
    for chunk in chunks
]
dataset = Dataset.from_list(training_data)

# 2. Configure LoRA adapter
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(model, peft_config)

# 3. Train Adapter
training_args = TrainingArguments(
    output_dir="./qwen_rag_finetuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    num_train_epochs=3,
    fp16=(device == "cuda"),
    optim="adamw_torch",
    save_strategy="no",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer
)

print("Starting LoRA Fine-Tuning...")
trainer.train()
model.eval()
print("Fine-tuning completed successfully!")